In [ ]:
#testing new method for checking if a transit occurs in a TESS sector, by directly reading the TSTART/TSTOP from the FITS files and comparing to the transit times. This should be more accurate than relying on the sector metadata, which can sometimes be incomplete or incorrect.   
#this script will loop through the confirmed planet list, check for the presence of FITS files in the TESS archive, read the TSTART/TSTOP from each FITS file, and determine if any transits occur within that time range. If so, it will extract the sector number from the FITS filename and save the relevant information to a new CSV file.
import os
import re
import pandas as pd

from astropy.io import fits


# ============================================================
# CONFIG
# ============================================================

PLANET_CSV = r"C:\Users\aasha\OneDrive\Desktop\correction\final_confirmed_planets.csv"

TESS_ARCHIVE = r"C:\Users\aasha\OneDrive\Desktop\correction\tess_archive"

OUTPUT_CSV = "confirmed_planet_usable_sectors.csv"


# ============================================================
# LOAD PLANET LIST
# ============================================================

planet_df = pd.read_csv(PLANET_CSV)

usable_rows = []


# ============================================================
# MAIN LOOP
# ============================================================

for _, row in planet_df.iterrows():

    try:

        tic_id = int(row["tic_id"])

        period = float(row["period"])

        # convert BJD -> BTJD
        t0 = float(row["t0"]) - 2457000

        tic_dir = os.path.join(
            TESS_ARCHIVE,
            f"TIC_{tic_id}"
        )

        if not os.path.exists(tic_dir):
            continue

        fits_files = [
            f for f in os.listdir(tic_dir)
            if f.endswith(".fits")
        ]

        for fits_name in fits_files:

            fits_path = os.path.join(
                tic_dir,
                fits_name
            )

            try:

                with fits.open(fits_path) as hdul:

                    hdr = hdul[0].header

                    tstart = float(hdr["TSTART"])
                    tstop = float(hdr["TSTOP"])

                # ----------------------------------------
                # CHECK IF ANY TRANSIT OCCURS
                # ----------------------------------------

                n_start = int((tstart - t0) // period) - 1
                n_end = int((tstop - t0) // period) + 1

                has_transit = False

                for n in range(n_start, n_end + 1):

                    transit_time = t0 + n * period

                    if tstart <= transit_time <= tstop:

                        has_transit = True
                        break

                if not has_transit:
                    continue

                # ----------------------------------------
                # EXTRACT SECTOR NUMBER
                # ----------------------------------------

                sector_match = re.search(
                    r"-s(\d+)-",
                    fits_name
                )

                sector = (
                    int(sector_match.group(1))
                    if sector_match
                    else -1
                )

                usable_rows.append({
                    "tic_id": tic_id,
                    "sector": sector,
                    "fits_file": fits_name,
                    "period": period,
                    "t0_btjd": t0,
                    "tstart": tstart,
                    "tstop": tstop
                })

            except Exception as e:

                print("FITS ERROR:", fits_name, e)

    except Exception as e:

        print("ROW ERROR:", e)


# ============================================================
# EXPORT
# ============================================================

usable_df = pd.DataFrame(usable_rows)

usable_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n===================================")
print("DONE")
print("===================================")

print("Usable sectors:", len(usable_df))

print("\nSaved:")
print(OUTPUT_CSV)


DONE
Usable sectors: 6268

Saved:
confirmed_planet_usable_sectors.csv


In [4]:
#checking distribution of cadence and usable points across the sectors, to see if there are any trends or outliers that might indicate data quality issues. This will help us understand if certain sectors have systematically worse data, which could affect our transit detection and characterization efforts. We can also look at the distribution of sector durations to see if there are any unusually short or long sectors that might impact our analysis.


import os
import numpy as np
import pandas as pd

from tqdm import tqdm
from astropy.io import fits


# ============================================================
# CONFIG
# ============================================================

TESS_ARCHIVE = r"C:\Users\aasha\OneDrive\Desktop\correction\tess_archive"

DATASETS = [
    ("final_confirmed_planets.csv", "confirmed"),
    ("final_toi_false_positives.csv", "toi_fp"),
    ("final_eclipsing_binaries.csv", "eb"),
    ("final_random_noisy_stars.csv", "random")
]

OUTPUT_CSV = "sector_statistics.csv"


# ============================================================
# STORAGE
# ============================================================

rows = []


# ============================================================
# MAIN
# ============================================================

for csv_path, dataset_name in DATASETS:

    print("\n===================================")
    print("PROCESSING:", dataset_name)
    print("===================================")

    df = pd.read_csv(csv_path)

    for _, row in tqdm(df.iterrows(), total=len(df)):

        try:

            tic_id = int(row["tic_id"])

            tic_dir = os.path.join(
                TESS_ARCHIVE,
                f"TIC_{tic_id}"
            )

            if not os.path.exists(tic_dir):
                continue

            fits_files = [
                f for f in os.listdir(tic_dir)
                if f.endswith(".fits")
            ]

            for fits_name in fits_files:

                fits_path = os.path.join(
                    tic_dir,
                    fits_name
                )

                try:

                    with fits.open(fits_path) as hdul:

                        # ------------------------------------------------
                        # HEADERS
                        # ------------------------------------------------

                        hdr0 = hdul[0].header
                        hdr1 = hdul[1].header

                        tstart = float(hdr0["TSTART"])
                        tstop = float(hdr0["TSTOP"])

                        sector_duration_days = (
                            tstop - tstart
                        )

                        # ------------------------------------------------
                        # TABLE DATA
                        # ------------------------------------------------

                        data = hdul[1].data

                        time = data["TIME"]
                        flux = data["PDCSAP_FLUX"]

                        # ------------------------------------------------
                        # RAW COUNTS
                        # ------------------------------------------------

                        total_points = len(time)

                        # ------------------------------------------------
                        # VALID MASK
                        # ------------------------------------------------

                        valid = (
                            np.isfinite(time) &
                            np.isfinite(flux)
                        )

                        usable_points = int(valid.sum())

                        # ------------------------------------------------
                        # CADENCE SPACING
                        # ------------------------------------------------

                        valid_time = time[valid]

                        if len(valid_time) > 1:

                            dt = np.diff(valid_time)

                            median_cadence_days = np.median(dt)

                            cadence_seconds = (
                                median_cadence_days * 86400
                            )

                        else:

                            cadence_seconds = np.nan

                        # ------------------------------------------------
                        # STORE
                        # ------------------------------------------------

                        rows.append({

                            "dataset": dataset_name,

                            "tic_id": tic_id,

                            "fits_file": fits_name,

                            "sector": int(hdr0["SECTOR"]),

                            "total_points": total_points,

                            "usable_points": usable_points,

                            "usable_fraction":
                                usable_points / total_points,

                            "cadence_seconds":
                                cadence_seconds,

                            "sector_duration_days":
                                sector_duration_days,

                            "tstart": tstart,

                            "tstop": tstop
                        })

                except Exception as e:

                    print(
                        "FITS ERROR:",
                        fits_name,
                        e
                    )

        except Exception as e:

            print(
                "ROW ERROR:",
                e
            )


# ============================================================
# EXPORT
# ============================================================

stats_df = pd.DataFrame(rows)

stats_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n===================================")
print("DONE")
print("===================================")

print("Total sectors:", len(stats_df))

print("\nSaved:")
print(OUTPUT_CSV)


# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("SUMMARY")
print("===================================")

print("\nCadence statistics (seconds):")
print(stats_df["cadence_seconds"].describe())

print("\nUsable points statistics:")
print(stats_df["usable_points"].describe())

print("\nSector duration statistics (days):")
print(stats_df["sector_duration_days"].describe())


PROCESSING: confirmed


100%|██████████| 1302/1302 [01:38<00:00, 13.28it/s]



PROCESSING: toi_fp


100%|██████████| 1333/1333 [02:40<00:00,  8.31it/s]



PROCESSING: eb


100%|██████████| 1400/1400 [04:31<00:00,  5.16it/s]



PROCESSING: random


100%|██████████| 1400/1400 [01:08<00:00, 20.42it/s] 



DONE
Total sectors: 25721

Saved:
sector_statistics.csv

SUMMARY

Cadence statistics (seconds):
count    25721.000000
mean       120.000234
std          0.002176
min        119.993388
25%        119.998739
50%        120.000081
75%        120.001297
max        120.012128
Name: cadence_seconds, dtype: float64

Usable points statistics:
count    25721.000000
mean     16033.469072
std       2610.997431
min       2504.000000
25%      14464.000000
50%      16262.000000
75%      17990.000000
max      30617.000000
Name: usable_points, dtype: float64

Sector duration statistics (days):
count    25721.000000
mean        26.540222
std          3.039852
min          7.229163
25%         25.490413
50%         26.485928
75%         27.163797
max         57.399714
Name: sector_duration_days, dtype: float64


In [ ]:
#making the final dataset with the resampled flux representations, and including the metadata about cadence, usable points, and sector duration for each sample. This will allow us to analyze how these factors correlate with the quality of the transit signals and the performance of our models. We can also use this information to filter or weight our samples during training, to improve the robustness of our results.
#it will produce two dataset with different resampling lengths (8192 and 16384), and include the metadata for each sample in the CSV files. The flux representations will be stored as JSON strings in the "flux" column, which can be parsed back into numpy arrays when loading the dataset for analysis or modeling.
import os
import json
import numpy as np
import pandas as pd

from tqdm import tqdm
from astropy.io import fits
from scipy.interpolate import interp1d


# ============================================================
# CONFIG
# ============================================================

TESS_ARCHIVE = r"C:\Users\aasha\OneDrive\Desktop\correction\tess_archive"

DATASETS = [
    ("final_confirmed_planets.csv", 1, "confirmed"),
    ("final_toi_false_positives.csv", 0, "toi_fp"),
    ("final_eclipsing_binaries.csv", 0, "eb"),
    ("final_random_noisy_stars.csv", 0, "random")
]

OUTPUT_8192 = "tess_dataset_8192.csv"
OUTPUT_16384 = "tess_dataset_16384.csv"

MAX_SECTORS_PER_TIC = 3


# ============================================================
# RESAMPLING
# ============================================================

def make_representation(time, flux, target_length):

    valid = (
        np.isfinite(time) &
        np.isfinite(flux)
    )

    time = time[valid]
    flux = flux[valid]

    if len(time) < 100:
        return None

    # --------------------------------------------
    # SORT
    # --------------------------------------------

    order = np.argsort(time)

    time = time[order]
    flux = flux[order]

    # --------------------------------------------
    # NORMALIZE
    # --------------------------------------------

    median_flux = np.median(flux)

    if median_flux == 0:
        return None

    flux = flux / median_flux

    # --------------------------------------------
    # FIXED GRID
    # --------------------------------------------

    new_time = np.linspace(
        time.min(),
        time.max(),
        target_length
    )

    try:

        interp = interp1d(
            time,
            flux,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate"
        )

        new_flux = interp(new_time)

    except Exception:

        return None

    return new_flux.astype(np.float32)


# ============================================================
# STORAGE
# ============================================================

rows_8192 = []
rows_16384 = []


# ============================================================
# MAIN
# ============================================================

for csv_path, label, dataset_name in DATASETS:

    print("\n===================================")
    print("PROCESSING:", dataset_name)
    print("===================================")

    df = pd.read_csv(csv_path)

    for _, row in tqdm(df.iterrows(), total=len(df)):

        try:

            tic_id = int(row["tic_id"])

            tic_dir = os.path.join(
                TESS_ARCHIVE,
                f"TIC_{tic_id}"
            )

            if not os.path.exists(tic_dir):
                continue

            fits_files = [
                f for f in os.listdir(tic_dir)
                if f.endswith(".fits")
            ]

            sector_rows = []

            # ------------------------------------------------
            # PROCESS EACH SECTOR
            # ------------------------------------------------

            for fits_name in fits_files:

                fits_path = os.path.join(
                    tic_dir,
                    fits_name
                )

                try:

                    with fits.open(fits_path) as hdul:

                        hdr0 = hdul[0].header

                        data = hdul[1].data

                        time = data["TIME"]
                        flux = data["PDCSAP_FLUX"]

                        total_points = len(time)

                        valid = (
                            np.isfinite(time) &
                            np.isfinite(flux)
                        )

                        usable_points = int(valid.sum())

                        if usable_points < 100:
                            continue

                        usable_fraction = (
                            usable_points / total_points
                        )

                        cadence_seconds = (
                            np.median(
                                np.diff(time[valid])
                            ) * 86400
                        )

                        sector_rows.append({

                            "tic_id":
                                tic_id,

                            "sector":
                                int(hdr0["SECTOR"]),

                            "dataset":
                                dataset_name,

                            "label":
                                label,

                            "fits_file":
                                fits_name,

                            "cadence_seconds":
                                cadence_seconds,

                            "total_points":
                                total_points,

                            "usable_points":
                                usable_points,

                            "usable_fraction":
                                usable_fraction,

                            "time":
                                time,

                            "flux":
                                flux
                        })

                except Exception as e:

                    print(
                        "FITS ERROR:",
                        fits_name,
                        e
                    )

            # ------------------------------------------------
            # TOP 3 SECTORS
            # ------------------------------------------------

            if len(sector_rows) == 0:
                continue

            sector_rows = sorted(
                sector_rows,
                key=lambda x: x["usable_fraction"],
                reverse=True
            )

            sector_rows = sector_rows[
                :MAX_SECTORS_PER_TIC
            ]

            # ------------------------------------------------
            # REPRESENTATIONS
            # ------------------------------------------------

            for srow in sector_rows:

                flux_8192 = make_representation(
                    srow["time"],
                    srow["flux"],
                    8192
                )

                flux_16384 = make_representation(
                    srow["time"],
                    srow["flux"],
                    16384
                )

                if (
                    flux_8192 is None or
                    flux_16384 is None
                ):
                    continue

                base_row = {

                    "tic_id":
                        srow["tic_id"],

                    "sector":
                        srow["sector"],

                    "dataset":
                        srow["dataset"],

                    "label":
                        srow["label"],

                    "fits_file":
                        srow["fits_file"],

                    "cadence_seconds":
                        srow["cadence_seconds"],

                    "total_points":
                        srow["total_points"],

                    "usable_points":
                        srow["usable_points"],

                    "usable_fraction":
                        srow["usable_fraction"]
                }

                row_8192 = dict(base_row)

                row_8192["flux"] = json.dumps(
                    flux_8192.tolist()
                )

                rows_8192.append(row_8192)

                row_16384 = dict(base_row)

                row_16384["flux"] = json.dumps(
                    flux_16384.tolist()
                )

                rows_16384.append(row_16384)

        except Exception as e:

            print(
                "ROW ERROR:",
                
            )


# ============================================================
# EXPORT
# ============================================================

df_8192 = pd.DataFrame(rows_8192)

df_16384 = pd.DataFrame(rows_16384)

df_8192.to_csv(
    OUTPUT_8192,
    index=False
)

df_16384.to_csv(
    OUTPUT_16384,
    index=False
)

print("\n===================================")
print("DONE")
print("===================================")

print("\n8192 samples:", len(df_8192))
print("16384 samples:", len(df_16384))

print("\nSaved:")
print(OUTPUT_8192)
print(OUTPUT_16384)


PROCESSING: confirmed


100%|██████████| 1302/1302 [04:19<00:00,  5.01it/s]



PROCESSING: toi_fp


100%|██████████| 1333/1333 [04:50<00:00,  4.59it/s]



PROCESSING: eb


100%|██████████| 1400/1400 [09:22<00:00,  2.49it/s]



PROCESSING: random


100%|██████████| 1400/1400 [01:48<00:00, 12.87it/s]



DONE

8192 samples: 9877
16384 samples: 9877

Saved:
tess_dataset_8192.csv
tess_dataset_16384.csv


In [6]:
a=pd.read_csv("tess_dataset_8192.csv")
a.head()

,tic_id,sector,dataset,label,fits_file,cadence_seconds,total_points,usable_points,usable_fraction,flux
0,709015,91,confirmed,1,tess2025099153000-s0091-0000000000709015-0288-...,119.997773,19780,10212,0.516279,"[0.9948985576629639, 0.994637668132782, 0.9933..."
1,1129033,31,confirmed,1,tess2020294194027-s0031-0000000001129033-0198-...,119.998096,18314,16170,0.882931,"[0.9998875856399536, 0.9994090795516968, 1.000..."
2,1129033,4,confirmed,1,tess2018292075959-s0004-0000000001129033-0124-...,119.998302,18684,14819,0.793139,"[0.999893307685852, 0.9996495842933655, 0.9996..."
3,2521105,91,confirmed,1,tess2025099153000-s0091-0000000002521105-0288-...,119.998658,19780,11235,0.567998,"[0.999825656414032, 0.9983697533607483, 1.0028..."
4,2764004,91,confirmed,1,tess2025099153000-s0091-0000000002764004-0288-...,119.999543,19780,11370,0.574823,"[0.9890503287315369, 1.0009032487869263, 0.990..."


In [7]:
a.label.value_counts()

label
0    6670
1    3207
Name: count, dtype: int64